# Setup

In [61]:
import time
from functools import partial
from itertools import product
from typing import NamedTuple

import jax
import jax.numpy as jnp
import jaxopt
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import scipy.optimize
import seaborn as sns
from jax import block_until_ready, jit, lax, pmap, random, vmap

In [62]:
class Algorithm(NamedTuple):
    name: str
    function: callable
    parameters: dict

class Benchmark(NamedTuple):
    name: str
    function: callable
    lower_bound: float
    upper_bound: float

class Result(NamedTuple):
    dimension: int
    benchmark: str
    algorithm: str
    best_fitness: float
    best_position: np.ndarray | jax.Array
    fitness_history: list[list[float]]
    execution_time: float

class SwarmState(NamedTuple):
    positions: np.ndarray
    velocities: np.ndarray
    personal_best_positions: np.ndarray
    personal_best_fitness: np.ndarray
    global_best_position: np.ndarray
    global_best_fitness: np.ndarray
    rng: np.random.Generator
    fitness_history: np.ndarray

class JaxSwarmState(NamedTuple):
    positions: jax.Array
    velocities: jax.Array
    personal_best_positions: jax.Array
    personal_best_fitness: jax.Array
    global_best_position: jax.Array
    global_best_fitness: jax.Array
    rng: random.PRNGKey

class GradientState(NamedTuple):
    current_pos: jax.Array

## PSO (NumPy)

In [63]:
def pso_numpy(
    seed: int,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    inertia_weight: float,
    cognitive_coeff: float,
    social_coeff: float,
    max_iterations: int,
) -> tuple:
    rng = np.random.default_rng(seed)

    initial_positions = rng.uniform(
        lower_bound, upper_bound, (n_particles, n_dimensions),
    )
    initial_velocities = np.zeros((n_particles, n_dimensions))
    initial_fitness = np.array(
        [objective_function(position) for position in initial_positions],
    )

    best_particle_idx = np.argmin(initial_fitness)
    global_best_position = initial_positions[best_particle_idx]
    global_best_fitness = initial_fitness[best_particle_idx]

    fitness_history = np.zeros(max_iterations)

    swarm_state = SwarmState(
        positions=initial_positions,
        velocities=initial_velocities,
        personal_best_positions=initial_positions,
        personal_best_fitness=initial_fitness,
        global_best_position=global_best_position,
        global_best_fitness=global_best_fitness,
        rng=rng,
        fitness_history=fitness_history,
    )

    for iteration in range(max_iterations):
        r1 = swarm_state.rng.random((n_particles, n_dimensions))
        r2 = swarm_state.rng.random((n_particles, n_dimensions))

        inertia_term = inertia_weight * swarm_state.velocities
        cognitive_term = (
            cognitive_coeff * r1 * (
                swarm_state.personal_best_positions - swarm_state.positions
            )
        )
        social_term = (
            social_coeff * r2 * (
                swarm_state.global_best_position - swarm_state.positions
            )
        )

        updated_velocities = inertia_term + cognitive_term + social_term
        updated_positions = swarm_state.positions + updated_velocities
        updated_positions = np.clip(updated_positions, lower_bound, upper_bound)

        updated_fitness = np.array(
            [objective_function(position) for position in updated_positions],
        )

        personal_improved = updated_fitness < swarm_state.personal_best_fitness
        improvement_mask = personal_improved[:, None]
        updated_personal_best_positions = np.where(
            improvement_mask,
            updated_positions,
            swarm_state.personal_best_positions,
        )
        updated_personal_best_fitness = np.where(
            personal_improved,
            updated_fitness,
            swarm_state.personal_best_fitness,
        )

        global_best_candidate_idx = np.argmin(updated_personal_best_fitness)
        global_best_candidate_fitness = updated_personal_best_fitness[
            global_best_candidate_idx
        ]
        global_improved = (
            global_best_candidate_fitness < swarm_state.global_best_fitness
        )

        updated_global_best_position = np.where(
            global_improved,
            updated_personal_best_positions[global_best_candidate_idx],
            swarm_state.global_best_position,
        )
        updated_global_best_fitness = np.where(
            global_improved,
            global_best_candidate_fitness,
            swarm_state.global_best_fitness,
        )

        updated_fitness_history = swarm_state.fitness_history
        updated_fitness_history[iteration] = updated_global_best_fitness

        swarm_state = SwarmState(
            positions=updated_positions,
            velocities=updated_velocities,
            personal_best_positions=updated_personal_best_positions,
            personal_best_fitness=updated_personal_best_fitness,
            global_best_position=updated_global_best_position,
            global_best_fitness=updated_global_best_fitness,
            rng=swarm_state.rng,
            fitness_history=updated_fitness_history,
        )

    return (
        swarm_state.global_best_position,
        swarm_state.global_best_fitness,
        swarm_state.fitness_history,
    )

## PSO (JAX)

In [64]:
@partial(
    jit,
    static_argnames=(
        "objective_function",
        "lower_bound",
        "upper_bound",
        "n_dimensions",
        "n_particles",
        "max_iterations",
    ),
)
def pso_jax(
    seed: random.PRNGKey,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    inertia_weight: float,
    cognitive_coeff: float,
    social_coeff: float,
    max_iterations: int,
    **_: any,
) -> tuple:
    position_key, velocity_key, state_key = random.split(seed, 3)

    search_range = upper_bound - lower_bound
    velocity_scale = 0.1
    velocity_limit = search_range * velocity_scale

    initial_positions = random.uniform(
        position_key,
        (n_particles, n_dimensions),
        minval=lower_bound,
        maxval=upper_bound,
    )
    initial_velocities = random.uniform(
        velocity_key,
        (n_particles, n_dimensions),
        minval=-velocity_limit,
        maxval=velocity_limit,
    )
    initial_fitness = vmap(objective_function)(initial_positions)

    best_particle_idx = jnp.argmin(initial_fitness)
    global_best_position = initial_positions[best_particle_idx]
    global_best_fitness = initial_fitness[best_particle_idx]

    swarm_state = JaxSwarmState(
        positions=initial_positions,
        velocities=initial_velocities,
        personal_best_positions=initial_positions,
        personal_best_fitness=initial_fitness,
        global_best_position=global_best_position,
        global_best_fitness=global_best_fitness,
        rng=state_key,
    )

    def update_step(swarm_state: JaxSwarmState, _: any) -> tuple:
        cognitive_key, social_key, next_key = random.split(swarm_state.rng, 3)
        r1 = random.uniform(cognitive_key, (n_particles, n_dimensions))
        r2 = random.uniform(social_key, (n_particles, n_dimensions))

        inertia_term = inertia_weight * swarm_state.velocities
        cognitive_term = (
            cognitive_coeff
            * r1
            * (swarm_state.personal_best_positions - swarm_state.positions)
        )
        social_term = (
            social_coeff
            * r2
            * (swarm_state.global_best_position - swarm_state.positions)
        )

        updated_velocities = inertia_term + cognitive_term + social_term
        updated_positions = swarm_state.positions + updated_velocities
        updated_positions = jnp.clip(updated_positions, lower_bound, upper_bound)

        updated_fitness = vmap(objective_function)(updated_positions)

        personal_improved = updated_fitness < swarm_state.personal_best_fitness

        updated_personal_best_positions = jnp.where(
            personal_improved[:, None],
            updated_positions,
            swarm_state.personal_best_positions,
        )
        updated_personal_best_fitness = jnp.where(
            personal_improved,
            updated_fitness,
            swarm_state.personal_best_fitness,
        )

        global_best_candidate_idx = jnp.argmin(updated_personal_best_fitness)
        global_best_candidate_position = updated_personal_best_positions[
            global_best_candidate_idx
        ]
        global_best_candidate_fitness = updated_personal_best_fitness[
            global_best_candidate_idx
        ]

        global_improved = (
            global_best_candidate_fitness < swarm_state.global_best_fitness
        )

        updated_global_best_position = jnp.where(
            global_improved,
            global_best_candidate_position,
            swarm_state.global_best_position,
        )
        updated_global_best_fitness = jnp.where(
            global_improved,
            global_best_candidate_fitness,
            swarm_state.global_best_fitness,
        )

        next_state = JaxSwarmState(
            positions=updated_positions,
            velocities=updated_velocities,
            personal_best_positions=updated_personal_best_positions,
            personal_best_fitness=updated_personal_best_fitness,
            global_best_position=updated_global_best_position,
            global_best_fitness=updated_global_best_fitness,
            rng=next_key,
        )

        return next_state, updated_global_best_fitness

    final_state, fitness_history = lax.scan(
        update_step,
        swarm_state,
        jnp.arange(max_iterations),
    )

    return (
        final_state.global_best_position,
        final_state.global_best_fitness,
        fitness_history,
    )

## PSOX (NumPy + SciPy)

In [65]:
def approximate_gradient(
    func: callable,
    position: np.ndarray,
    epsilon: float = 1e-8,
) -> np.ndarray:
    grad = np.zeros_like(position)
    for i in range(len(position)):
        pos_plus = position.copy()
        pos_minus = position.copy()

        pos_plus[i] += epsilon
        pos_minus[i] -= epsilon

        grad[i] = (func(pos_plus) - func(pos_minus)) / (2 * epsilon)

    return grad


def optimize_adam_numpy(
    objective_function: callable,
    initial_position: np.ndarray,
    lower_bound: float,
    upper_bound: float,
    n_local_optimization_steps: int,
    learning_rate: float,
    beta1: float = 0.9,
    beta2: float = 0.999,
    epsilon: float = 1e-8,
    grad_epsilon: float = 1e-8,
) -> np.ndarray:
    theta = initial_position.copy()
    m = np.zeros_like(theta)
    v = np.zeros_like(theta)

    for t in range(1, n_local_optimization_steps + 1):
        g = approximate_gradient(objective_function, theta, grad_epsilon)

        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * (g ** 2)

        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)

        theta = theta - learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)
        theta = np.clip(theta, lower_bound, upper_bound)

    return theta


def local_optimize(
    objective_function: callable,
    initial_position: np.ndarray,
    lower_bound: float,
    upper_bound: float,
    n_local_optimization_steps: int,
    learning_rate: float,
) -> tuple:
    best_position = initial_position.copy()
    best_fitness = objective_function(initial_position)

    bounds = [(lower_bound, upper_bound)] * len(initial_position)

    lbfgs_result = scipy.optimize.minimize(
        objective_function,
        initial_position,
        method="L-BFGS-B",
        bounds=bounds,
    )

    lbfgs_candidate = np.clip(lbfgs_result.x, lower_bound, upper_bound)
    lbfgs_fitness = objective_function(lbfgs_candidate)

    if lbfgs_fitness < best_fitness:
        best_position = lbfgs_candidate
        best_fitness = lbfgs_fitness

    cg_result = scipy.optimize.minimize(
        objective_function,
        initial_position,
        method="CG",
    )

    cg_candidate = np.clip(cg_result.x, lower_bound, upper_bound)
    cg_fitness = objective_function(cg_candidate)

    if cg_fitness < best_fitness:
        best_position = cg_candidate
        best_fitness = cg_fitness

    adam_candidate = optimize_adam_numpy(
        objective_function,
        initial_position,
        lower_bound,
        upper_bound,
        n_local_optimization_steps,
        learning_rate,
    )

    adam_fitness = objective_function(adam_candidate)

    if adam_fitness < best_fitness:
        best_position = adam_candidate
        best_fitness = adam_fitness

    return best_position, best_fitness


def psox_numpy(
    seed: int,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    inertia_weight: float,
    cognitive_coeff: float,
    social_coeff: float,
    max_iterations: int,
    local_optimization_interval: int,
    n_local_optimization_steps: int,
    learning_rate: float,
    **_: any,
) -> tuple:
    rng = np.random.default_rng(seed)

    initial_positions = rng.uniform(
        lower_bound,
        upper_bound,
        (n_particles, n_dimensions),
    )
    initial_velocities = np.zeros((n_particles, n_dimensions))
    initial_fitness = np.array(
        [objective_function(position) for position in initial_positions],
    )

    best_particle_idx = np.argmin(initial_fitness)
    global_best_position = initial_positions[best_particle_idx]
    global_best_fitness = initial_fitness[best_particle_idx]

    fitness_history = np.zeros(max_iterations)

    swarm_state = SwarmState(
        positions=initial_positions,
        velocities=initial_velocities,
        personal_best_positions=initial_positions,
        personal_best_fitness=initial_fitness,
        global_best_position=global_best_position,
        global_best_fitness=global_best_fitness,
        rng=rng,
        fitness_history=fitness_history,
    )

    for i in range(max_iterations):
        r1 = swarm_state.rng.random((n_particles, n_dimensions))
        r2 = swarm_state.rng.random((n_particles, n_dimensions))

        inertia_term = inertia_weight * swarm_state.velocities
        cognitive_term = (
            cognitive_coeff
            * r1
            * (swarm_state.personal_best_positions - swarm_state.positions)
        )
        social_term = (
            social_coeff
            * r2
            * (swarm_state.global_best_position - swarm_state.positions)
        )

        updated_velocities = inertia_term + cognitive_term + social_term
        updated_positions = swarm_state.positions + updated_velocities
        updated_positions = np.clip(updated_positions, lower_bound, upper_bound)

        updated_fitness = np.array(
            [objective_function(pos) for pos in updated_positions],
        )

        personal_improved = updated_fitness < swarm_state.personal_best_fitness
        updated_personal_best_positions = np.where(
            personal_improved[:, None],
            updated_positions,
            swarm_state.personal_best_positions,
        )
        updated_personal_best_fitness = np.where(
            personal_improved,
            updated_fitness,
            swarm_state.personal_best_fitness,
        )

        global_best_candidate_idx = np.argmin(updated_personal_best_fitness)
        global_best_candidate_fitness = updated_personal_best_fitness[
            global_best_candidate_idx
        ]
        global_improved = (
            global_best_candidate_fitness < swarm_state.global_best_fitness
        )

        global_best_candidate_position = np.where(
            global_improved,
            updated_personal_best_positions[global_best_candidate_idx],
            swarm_state.global_best_position,
        )
        updated_global_best_fitness = np.where(
            global_improved,
            global_best_candidate_fitness,
            swarm_state.global_best_fitness,
        )

        if i % local_optimization_interval == 0:
            local_opt_position, local_opt_fitness = local_optimize(
                objective_function,
                global_best_candidate_position,
                lower_bound,
                upper_bound,
                n_local_optimization_steps,
                learning_rate,
            )

            local_opt_improved = local_opt_fitness < updated_global_best_fitness
            updated_global_best_position = (
                local_opt_position
                if local_opt_improved
                else global_best_candidate_position
            )
            updated_global_best_fitness = (
                local_opt_fitness
                if local_opt_improved
                else updated_global_best_fitness
            )

            if local_opt_improved and (
                updated_global_best_fitness < swarm_state.global_best_fitness
            ):
                updated_personal_best_positions[global_best_candidate_idx] = (
                    updated_global_best_position
                )
                updated_personal_best_fitness[global_best_candidate_idx] = (
                    updated_global_best_fitness
                )
        else:
            updated_global_best_position = global_best_candidate_position

        updated_history = swarm_state.fitness_history
        updated_history[i] = updated_global_best_fitness

        swarm_state = SwarmState(
            positions=updated_positions,
            velocities=updated_velocities,
            personal_best_positions=updated_personal_best_positions,
            personal_best_fitness=updated_personal_best_fitness,
            global_best_position=updated_global_best_position,
            global_best_fitness=updated_global_best_fitness,
            rng=swarm_state.rng,
            fitness_history=updated_history,
        )

    return (
        swarm_state.global_best_position,
        swarm_state.global_best_fitness,
        swarm_state.fitness_history,
    )

## PSOX (JAX + JAXopt)

In [66]:
@partial(
    jit,
    static_argnames=(
        "objective_function",
        "lower_bound",
        "upper_bound",
        "n_dimensions",
        "n_particles",
        "max_iterations",
        "stagnation_threshold",
        "n_local_optimization_steps",
        "learning_rate",
    ),
)
def psox_jax(
    seed: random.PRNGKey,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    inertia_weight: float,
    social_coeff: float,
    cognitive_coeff: float,
    max_iterations: int,
    stagnation_threshold: int,
    n_local_optimization_steps: int,
    learning_rate: float,
    **_: any,
) -> tuple:
    key = seed
    lower = jnp.array(lower_bound)
    upper = jnp.array(upper_bound)
    position_key, velocity_key, state_key = random.split(key, 3)

    search_range = upper - lower
    velocity_scale = 0.1
    velocity_limit = search_range * velocity_scale

    initial_positions = random.uniform(
        position_key,
        (n_particles, n_dimensions),
        minval=lower,
        maxval=upper,
    )
    initial_velocities = random.uniform(
        velocity_key,
        (n_particles, n_dimensions),
        minval=-velocity_limit,
        maxval=velocity_limit,
    )
    initial_fitness = vmap(objective_function)(initial_positions)

    best_particle_idx = jnp.argmin(initial_fitness)
    global_best_position = initial_positions[best_particle_idx]
    global_best_fitness = initial_fitness[best_particle_idx]

    initial_state = JaxSwarmState(
        positions=initial_positions,
        velocities=initial_velocities,
        personal_best_positions=initial_positions,
        personal_best_fitness=initial_fitness,
        global_best_position=global_best_position,
        global_best_fitness=global_best_fitness,
        rng=state_key,
    )

    lbfgs_solver = jaxopt.LBFGS(
        fun=objective_function,
        maxiter=n_local_optimization_steps,
        implicit_diff=False,
    )
    adam_solver = jaxopt.OptaxSolver(
        fun=objective_function,
        opt=optax.adam(learning_rate=learning_rate),
        maxiter=n_local_optimization_steps,
        implicit_diff=False,
    )
    nonlinear_cg_solver = jaxopt.NonlinearCG(
        fun=objective_function,
        maxiter=n_local_optimization_steps,
        implicit_diff=False,
    )

    def update_step(carry: tuple, _: int) -> tuple:
        swarm_state, stagnation_counter = carry

        cognitive_key, social_key, next_key = random.split(swarm_state.rng, 3)
        r1 = random.uniform(cognitive_key, (n_particles, n_dimensions))
        r2 = random.uniform(social_key, (n_particles, n_dimensions))

        inertia_term = inertia_weight * swarm_state.velocities
        cognitive_term = (
            cognitive_coeff
            * r1
            * (swarm_state.personal_best_positions - swarm_state.positions)
        )
        social_term = (
            social_coeff
            * r2
            * (swarm_state.global_best_position - swarm_state.positions)
        )

        updated_velocities = inertia_term + cognitive_term + social_term
        updated_positions = swarm_state.positions + updated_velocities
        updated_positions = jnp.clip(updated_positions, lower, upper)

        updated_fitness = vmap(objective_function)(updated_positions)

        personal_improved = updated_fitness < swarm_state.personal_best_fitness
        updated_personal_best_positions = jnp.where(
            personal_improved[:, None],
            updated_positions,
            swarm_state.personal_best_positions,
        )
        updated_personal_best_fitness = jnp.where(
            personal_improved,
            updated_fitness,
            swarm_state.personal_best_fitness,
        )

        global_best_candidate_idx = jnp.argmin(updated_personal_best_fitness)
        global_best_candidate_fitness = updated_personal_best_fitness[
            global_best_candidate_idx
        ]
        global_improved = (
            global_best_candidate_fitness < swarm_state.global_best_fitness
        )

        global_best_candidate_position = jnp.where(
            global_improved,
            updated_personal_best_positions[global_best_candidate_idx],
            swarm_state.global_best_position,
        )
        updated_global_best_fitness = jnp.where(
            global_improved,
            global_best_candidate_fitness,
            swarm_state.global_best_fitness,
        )

        def apply_local_optimization(_: None) -> tuple:
            lbfgs_position, _ = lbfgs_solver.run(global_best_candidate_position)
            adam_position, _ = adam_solver.run(global_best_candidate_position)
            ncg_position, _ = nonlinear_cg_solver.run(global_best_candidate_position)

            lbfgs_position = jnp.clip(lbfgs_position, lower, upper)
            adam_position = jnp.clip(adam_position, lower, upper)
            ncg_position = jnp.clip(ncg_position, lower, upper)

            final_lbfgs_fitness = objective_function(lbfgs_position)
            final_adam_fitness = objective_function(adam_position)
            final_ncg_fitness = objective_function(ncg_position)

            best_fitness = jnp.minimum(
                jnp.minimum(final_lbfgs_fitness, final_adam_fitness),
                final_ncg_fitness,
            )
            best_position = jnp.where(
                best_fitness == final_lbfgs_fitness,
                lbfgs_position,
                jnp.where(
                    best_fitness == final_adam_fitness,
                    adam_position,
                    ncg_position,
                ),
            )
            return best_position, best_fitness

        def skip_local_optimization(_: None) -> tuple:
            return global_best_candidate_position, updated_global_best_fitness

        local_opt_best_position, local_opt_best_fitness = lax.cond(
            stagnation_counter >= stagnation_threshold,
            apply_local_optimization,
            skip_local_optimization,
            None,
        )

        local_opt_improved = local_opt_best_fitness < updated_global_best_fitness
        updated_global_best_position = jnp.where(
            local_opt_improved,
            local_opt_best_position,
            global_best_candidate_position,
        )
        updated_global_best_fitness = jnp.where(
            local_opt_improved,
            local_opt_best_fitness,
            updated_global_best_fitness,
        )

        any_improvement = updated_global_best_fitness < swarm_state.global_best_fitness
        winner_mask = (jnp.arange(n_particles) == global_best_candidate_idx)[:, None]
        should_update_mask = (local_opt_improved & any_improvement) & winner_mask

        final_personal_best_positions = jnp.where(
            should_update_mask,
            updated_global_best_position,
            updated_personal_best_positions,
        )
        final_personal_best_fitness = jnp.where(
            (local_opt_improved & any_improvement)
            & (jnp.arange(n_particles) == global_best_candidate_idx),
            updated_global_best_fitness,
            updated_personal_best_fitness,
        )

        next_state = JaxSwarmState(
            positions=updated_positions,
            velocities=updated_velocities,
            personal_best_positions=final_personal_best_positions,
            personal_best_fitness=final_personal_best_fitness,
            global_best_position=updated_global_best_position,
            global_best_fitness=updated_global_best_fitness,
            rng=next_key,
        )

        next_stagnation_counter = jnp.where(
            any_improvement,
            jnp.int32(0),
            stagnation_counter + jnp.int32(1),
        )

        return (next_state, next_stagnation_counter), updated_global_best_fitness

    (final_state, _), fitness_history = lax.scan(
        update_step,
        (initial_state, jnp.int32(0)),
        jnp.arange(max_iterations),
    )

    return (
        final_state.global_best_position,
        final_state.global_best_fitness,
        fitness_history,
    )

In [67]:
def parallel_psox_jax(
    seeds: random.PRNGKey,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    inertia_weight: float,
    cognitive_coeff: float,
    social_coeff: float,
    max_iterations: int,
    local_optimization_interval: int,
    n_local_optimization_steps: int,
    learning_rate: float,
) -> tuple[jnp.ndarray, float, jnp.ndarray]:
    psox_partial = partial(
        psox_jax,
        objective_function=objective_function,
        lower_bound=lower_bound,
        upper_bound=upper_bound,
        n_dimensions=n_dimensions,
        n_particles=n_particles,
        inertia_weight=inertia_weight,
        cognitive_coeff=cognitive_coeff,
        social_coeff=social_coeff,
        max_iterations=max_iterations,
        local_optimization_interval=local_optimization_interval,
        n_local_optimization_steps=n_local_optimization_steps,
        learning_rate=learning_rate,
    )

    best_positions, best_fitnesses, fitness_histories = pmap(psox_partial)(seeds)

    best_idx = jnp.argmin(best_fitnesses)
    return (
        best_positions[best_idx],
        best_fitnesses[best_idx],
        fitness_histories[best_idx],
    )

# Visualizations

In [68]:
def _save_figure(fig: plt.Figure, filename: str, config: dict) -> None:
    save_path = config["output_path"] / filename
    fig.savefig(save_path, bbox_inches="tight", format="pdf", dpi=300)
    plt.close(fig)


def plot_execution_time(df: pd.DataFrame, config: dict) -> None:
    benchmarks = df["Benchmark"].unique()
    algorithms = df["Algorithm"].unique()
    dimensions = df["Dimension"].unique()

    colors = ["#909090", "#0056AD", "#6D87A1", "#0080FF"]

    plt.rcParams.update(
        {
            "font.size": 11,
            "axes.titlesize": 11,
            "axes.labelsize": 11,
            "xtick.labelsize": 11,
            "ytick.labelsize": 11,
            "legend.fontsize": 11,
        },
    )

    fig, axes = plt.subplots(2, 2, figsize=(5, 4), sharex=True, sharey=True)
    axes_flattened = axes.flatten()

    lines = []
    labels = []

    for idx_ax, (ax, benchmark) in enumerate(zip(axes_flattened, benchmarks)):
        df_benchmark = df[df["Benchmark"] == benchmark]

        for idx_algo, algorithm in enumerate(algorithms):
            df_subset = df_benchmark[df_benchmark["Algorithm"] == algorithm]
            mean = df_subset["Mean of Execution Times (s)"]
            std = df_subset["Standard Deviation of Execution Times (s)"]

            (line,) = ax.plot(
                dimensions,
                mean,
                label=algorithm,
                marker="o",
                color=colors[idx_algo],
                markersize=6,
                linewidth=2,
            )
            ax.fill_between(
                dimensions, mean - std, mean + std, color=colors[idx_algo], alpha=0.2,
            )

            if idx_ax == 0:
                lines.append(line)
                labels.append(algorithm)

        ax.set_yscale("log")
        ax.set_title(f"{benchmark}")
        sns.despine(ax=ax, left=True)

    fig.supxlabel("Dimensão")
    fig.supylabel("Tempo de Execução (s)")

    fig.legend(
        lines,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.05),
        ncol=len(algorithms),
        frameon=False,
    )

    plt.tight_layout()
    _save_figure(fig, "execution_time_plot.pdf", config)


def plot_convergence(df: pd.DataFrame, config: dict) -> None:
    plt.rcParams.update(
        {
            "font.size": 13,
            "axes.titlesize": 13,
            "axes.labelsize": 13,
            "xtick.labelsize": 13,
            "ytick.labelsize": 13,
            "legend.fontsize": 13,
        },
    )

    benchmarks = df["Benchmark"].unique()
    dimensions = df["Dimension"].unique()
    algorithms = df["Algorithm"].unique()

    colors = ["#909090", "#0056AD", "#6D87A1", "#0080FF"]

    fig, axes = plt.subplots(4, 4, figsize=(12, 12), sharex=True)

    lines = []
    labels = []

    for i, benchmark in enumerate(benchmarks):
        for j, dimension in enumerate(dimensions):
            ax = axes[i, j]
            df_subset = df[
                (df["Dimension"] == dimension) & (df["Benchmark"] == benchmark)
            ]

            for k, (_, row) in enumerate(df_subset.iterrows()):
                mean_history = jnp.array(row["Mean Fitness History"])
                std_history = jnp.array(row["Std Fitness History"])
                iterations = range(len(mean_history))

                (line,) = ax.plot(
                    iterations,
                    mean_history,
                    label=row["Algorithm"],
                    color=colors[k],
                    linewidth=2,
                )
                ax.fill_between(
                    iterations,
                    mean_history - std_history,
                    mean_history + std_history,
                    alpha=0.2,
                    color=colors[k],
                )

                if i == 0 and j == 0:
                    lines.append(line)
                    labels.append(row["Algorithm"])

            sns.despine(ax=ax, left=True)
            ax.set_title(f"{benchmark} (d = {dimension})")

    fig.supxlabel("Iteração")
    fig.supylabel("Fitness")

    fig.legend(
        lines,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.03),
        ncol=len(algorithms),
        frameon=False,
    )

    plt.tight_layout()
    _save_figure(fig, "convergence_plot.pdf", config)


def _generate_comparison_table(
    df: pd.DataFrame,
    config: dict,
    mean_col: str,
    std_col: str,
    output_filename: str,
    caption: str,
    label: str,
) -> None:
    df_proc = df.copy()

    df_proc["formatted"] = (
        df_proc[mean_col].map("{:.2e}".format)
        + r" $\pm$ "
        + df_proc[std_col].map("{:.2e}".format)
    )

    df_pivot = df_proc.pivot_table(
        index=["Benchmark", "Dimension"],
        columns="Algorithm",
        values=["formatted", mean_col],
        aggfunc="first",
    )

    means = df_pivot[mean_col].fillna(float("inf"))

    display_df = df_pivot["formatted"].copy()

    for idx in display_df.index:
        row_means = means.loc[idx]
        best_algo = row_means.idxmin()
        if pd.notna(display_df.loc[idx, best_algo]):
            display_df.loc[idx, best_algo] = (
                r"\textbf{" + display_df.loc[idx, best_algo] + "}"
            )

    display_df = display_df.reset_index()

    display_df["Benchmark"] = display_df["Benchmark"].apply(
        lambda x: f"\\textit{{{x}}}",
    )

    for col in display_df.columns:
        if col not in ["Benchmark", "Dimension"]:
            display_df = display_df.rename(columns={col: f"\\textit{{{col}}}"})

    output_path = config["output_path"] / output_filename

    num_algo_cols = len(display_df.columns) - 2
    column_format = "ll" + "c" * num_algo_cols

    latex_code = display_df.style.hide(axis="index").to_latex(
        column_format=column_format,
        hrules=True,
        caption=caption,
        label=label,
        position="h",
    )

    with output_path.open("w") as f:
        f.write(latex_code)


def create_convergence_table(df: pd.DataFrame, config: dict) -> None:
    _generate_comparison_table(
        df=df,
        config=config,
        mean_col="Mean of Fitness",
        std_col="Standard Deviation of Fitness",
        output_filename="convergence_table.tex",
        caption=r"Convergence comparison (Mean Fitness $\pm$ Std Dev).",
        label="tab:convergence",
    )


def create_execution_time_table(df: pd.DataFrame, config: dict) -> None:
    _generate_comparison_table(
        df=df,
        config=config,
        mean_col="Mean of Execution Times (s)",
        std_col="Standard Deviation of Execution Times (s)",
        output_filename="execution_time_table.tex",
        caption="Execution time comparison in seconds.",
        label="tab:execution_time",
    )


def generate_visualizations(df: pd.DataFrame, config: dict) -> None:
    print("Plotting convergence...")
    plot_convergence(df, config)

    print("Plotting execution time...")
    plot_execution_time(df, config)

    print("Creating convergence table (LaTeX)...")
    create_convergence_table(df, config)

    print("Creating execution time table (LaTeX)...")
    create_execution_time_table(df, config)

# Benchmark Functions

In [69]:
def _get_xp(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    return jnp if isinstance(x, jax.Array) else np


def ackley(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    n = x.shape[0]
    sum_sq = xp.sum(x**2)
    sum_cos = xp.sum(xp.cos(2 * xp.pi * x))
    return (
        -20 * xp.exp(-0.2 * xp.sqrt(sum_sq / n))
        - xp.exp(sum_cos / n)
        + 20
        + xp.e
    )


def griewank(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    n = x.shape[0]
    i = xp.arange(1, n + 1)
    return xp.sum(x**2) / 4000 - xp.prod(xp.cos(x / xp.sqrt(i))) + 1


def levy(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    w = 1 + (x - 1) / 4
    term1 = xp.sin(xp.pi * w[0]) ** 2
    term2 = xp.sum((w[:-1] - 1) ** 2 * (1 + 10 * xp.sin(xp.pi * w[:-1] + 1) ** 2))
    term3 = (w[-1] - 1) ** 2 * (1 + xp.sin(2 * xp.pi * w[-1]) ** 2)
    return term1 + term2 + term3


def rastrigin(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    n = x.shape[0]
    return 10 * n + xp.sum(x**2 - 10 * xp.cos(2 * xp.pi * x))


def rosenbrock(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    return xp.sum(100 * (x[1:] - x[:-1] ** 2) ** 2 + (1 - x[:-1]) ** 2)


def schwefel(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    n = x.shape[0]
    return 418.9829 * n - xp.sum(x * xp.sin(xp.sqrt(xp.abs(x))))


def sphere(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    return xp.sum(x**2)


def styblinski_tang(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    return xp.sum(x**4 - 16 * x**2 + 5 * x) / 2


def zakharov(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    n = x.shape[0]
    i = xp.arange(1, n + 1)
    s = xp.sum(0.5 * i * x)
    return xp.sum(x**2) + s**2 + s**4

# Configuration

In [70]:
parameters = {
    "n_particles": 30,
    "max_iterations": 100,
    "cognitive_coeff": 1.5,
    "social_coeff": 1.5,
    "inertia_weight": 0.7,
}

extra_parameters = {
    "stagnation_threshold": 5,
    "local_optimization_interval": 10,
    "n_local_optimization_steps": 5,
    "learning_rate": 0.001,
}

algorithms = [
    # Algorithm("PSO (NumPy)", pso_numpy, parameters),
    Algorithm("PSO (JAX)", pso_jax, parameters),
    # Algorithm("PSOX (NumPy)", psox_numpy, {**parameters, **extra_parameters}),
    Algorithm("PSOX (JAX)", psox_jax, {**parameters, **extra_parameters}),
]

benchmarks = [
    Benchmark("Ackley", ackley, -32.768, 32.768),
    Benchmark("Griewank", griewank, -600, 600),
    Benchmark("Levy", levy, -10, 10),
    Benchmark("Rastrigin", rastrigin, -5.12, 5.12),
    Benchmark("Rosenbrock", rosenbrock, -5, 10),
    Benchmark("Schwefel", schwefel, -500, 500),
    Benchmark("Sphere", sphere, -5.12, 5.12),
    Benchmark("Styblinski-Tang", styblinski_tang, -5, 5),
    Benchmark("Zakharov", zakharov, -5.0, 10.0),
]

dimensions = [10, 30, 50, 100]

n_runs = 50

# Experiments

In [71]:
def run_experiments(
    algorithms: list[Algorithm],
    benchmarks: list[Benchmark],
    dimensions: list[int],
    n_runs: int,
) -> list[Result]:
    results = []
    for (algorithm, dimension, benchmark, run) in product(
        algorithms, dimensions, benchmarks, range(1, n_runs + 1),
    ):
        print(f"{algorithm.name} | {benchmark.name} | {dimension} | {run}/{n_runs}")

        if algorithm.name == "Parallel PSOX (JAX)":
            seed = random.split(random.PRNGKey(run), jax.local_device_count())

            algorithm.function(
                seed,
                benchmark.function,
                benchmark.lower_bound,
                benchmark.upper_bound,
                dimension,
                algorithm.parameters["n_particles"],
                algorithm.parameters["inertia_weight"],
                algorithm.parameters["cognitive_coeff"],
                algorithm.parameters["social_coeff"],
                algorithm.parameters["max_iterations"],
                algorithm.parameters["local_optimization_interval"],
                algorithm.parameters["n_local_optimization_steps"],
                algorithm.parameters["learning_rate"],
            )

            start = time.perf_counter()

            result = algorithm.function(
                seed,
                benchmark.function,
                benchmark.lower_bound,
                benchmark.upper_bound,
                dimension,
                algorithm.parameters["n_particles"],
                algorithm.parameters["inertia_weight"],
                algorithm.parameters["cognitive_coeff"],
                algorithm.parameters["social_coeff"],
                algorithm.parameters["max_iterations"],
                algorithm.parameters["local_optimization_interval"],
                algorithm.parameters["n_local_optimization_steps"],
                algorithm.parameters["learning_rate"],
            )
            block_until_ready(result)

            position, fitness, history = result

        elif algorithm.name in ["PSO (JAX)", "PSOX (JAX)"]:
            seed = random.PRNGKey(run)

            algorithm.function(
                seed,
                benchmark.function,
                benchmark.lower_bound,
                benchmark.upper_bound,
                dimension,
                **algorithm.parameters,
            )

            start = time.perf_counter()

            result = algorithm.function(
                seed,
                benchmark.function,
                benchmark.lower_bound,
                benchmark.upper_bound,
                dimension,
                **algorithm.parameters,
            )
            block_until_ready(result)

            position, fitness, history = result

        else:
            seed = run

            start = time.perf_counter()

            position, fitness, history = algorithm.function(
                seed,
                benchmark.function,
                benchmark.lower_bound,
                benchmark.upper_bound,
                dimension,
                **algorithm.parameters,
            )

        end = time.perf_counter()

        results.append(
            Result(
                dimension=dimension,
                benchmark=benchmark.name,
                algorithm=algorithm.name,
                best_fitness=float(fitness),
                best_position=position.tolist(),
                fitness_history=history.tolist(),
                execution_time=end - start,
            ),
        )

    return results

In [72]:
results = run_experiments(algorithms, benchmarks, dimensions, n_runs)
pd.DataFrame(results).to_csv("../data/experiments_results.csv", index=False)

PSO (JAX) | Ackley | 10 | 1/50
PSO (JAX) | Ackley | 10 | 2/50
PSO (JAX) | Ackley | 10 | 3/50
PSO (JAX) | Ackley | 10 | 4/50
PSO (JAX) | Ackley | 10 | 5/50
PSO (JAX) | Ackley | 10 | 6/50
PSO (JAX) | Ackley | 10 | 7/50
PSO (JAX) | Ackley | 10 | 8/50
PSO (JAX) | Ackley | 10 | 9/50
PSO (JAX) | Ackley | 10 | 10/50
PSO (JAX) | Ackley | 10 | 11/50
PSO (JAX) | Ackley | 10 | 12/50
PSO (JAX) | Ackley | 10 | 13/50
PSO (JAX) | Ackley | 10 | 14/50
PSO (JAX) | Ackley | 10 | 15/50
PSO (JAX) | Ackley | 10 | 16/50
PSO (JAX) | Ackley | 10 | 17/50
PSO (JAX) | Ackley | 10 | 18/50
PSO (JAX) | Ackley | 10 | 19/50
PSO (JAX) | Ackley | 10 | 20/50
PSO (JAX) | Ackley | 10 | 21/50
PSO (JAX) | Ackley | 10 | 22/50
PSO (JAX) | Ackley | 10 | 23/50
PSO (JAX) | Ackley | 10 | 24/50
PSO (JAX) | Ackley | 10 | 25/50
PSO (JAX) | Ackley | 10 | 26/50
PSO (JAX) | Ackley | 10 | 27/50
PSO (JAX) | Ackley | 10 | 28/50
PSO (JAX) | Ackley | 10 | 29/50
PSO (JAX) | Ackley | 10 | 30/50
PSO (JAX) | Ackley | 10 | 31/50
PSO (JAX) | Ackle